# 9.1 · 感知器 & 多层感知器 / Perceptron & MLP

> **课程定位 / Where this fits**
> 第 1 课，**Part 9 · 深度学习基础**。
> Lesson 1, **Part 9 · Deep Learning Foundations**.
>
> 深度学习的起点是 1958 年的**感知器**——一个最简单的"人工神经元"。但它有个致命缺陷：**连 XOR 都学不会**（这个发现一度让神经网络研究停滞了十几年）。解药是**多层感知器(MLP)**：堆叠隐藏层 + 非线性激活，就能拟合任意函数。理解"为什么需要隐藏层和非线性"，是理解整个深度学习的第一步。
> Deep learning starts with the 1958 **perceptron** — the simplest "artificial neuron". But it has a fatal flaw: **it can't even learn XOR** (a discovery that stalled neural-net research for over a decade). The cure is the **multilayer perceptron (MLP)**: stacked hidden layers + nonlinear activations can fit any function. Understanding "why we need hidden layers and nonlinearity" is the first step into all of deep learning.
>
> 💼 **实战/面试视角**："感知器为什么学不会 XOR / 为什么要激活函数 / MLP 怎么解决" 是深度学习入门必考。
> 💼 **Practical/interview angle:** "why can't a perceptron learn XOR / why activations / how MLP fixes it" — DL fundamentals.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $\mathbf{w}, b$ —— 权重向量与偏置 / weights and bias
> - $\sigma$ —— 激活函数 / activation function
> - 一个神经元 neuron: $a = \sigma(\mathbf{w}^\top\mathbf{x}+b)$

> 💡 **面试相关 / Interview-relevant**
> - "感知器为什么解决不了 XOR（线性不可分）"（出镜率 ★★★★★）
> - "为什么神经网络需要非线性激活函数"（出镜率 ★★★★★）
> - "MLP 为什么能拟合任意函数（万能逼近）"（★★★★）
> - "一个神经元在做什么"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解单个感知器 = 线性分类器，及其学习规则。
   Understand a single perceptron = a linear classifier, and its learning rule.
2. 亲眼看到感知器**学不会 XOR**（线性不可分）。
   See the perceptron **fail on XOR** (not linearly separable).
3. 理解**隐藏层 + 非线性激活**为何能解决 XOR。
   Understand why hidden layers + nonlinear activations solve XOR.
4. 理解"去掉激活函数，深层网络退化成线性"。
   Understand "without activations, a deep net collapses to linear".
5. 用 MLP 跑一个真实多分类（Digits）。
   Run an MLP on a real multi-class task (Digits).

## 目录 / TOC
1. [先建直觉：一个神经元 ⭐](#1)
2. [感知器学不会 XOR ⭐](#2)
3. [MLP 解决 XOR ⭐](#3)
4. [为什么必须非线性激活 ⭐](#4)
5. [🔢 MLP on Digits + 小结](#5)


<a id="1"></a>
## 1. 先建直觉：一个神经元 ⭐ / Intuition: One Neuron

一个**人工神经元**做的事极简单：把输入加权求和、加偏置、再过一个**激活函数**：
An **artificial neuron** does something very simple: weighted-sum the inputs, add a bias, then pass through an **activation function**:

$$a = \sigma(\mathbf{w}^\top\mathbf{x} + b) = \sigma(w_1 x_1 + \dots + w_d x_d + b)$$

如果激活是阶跃函数（>0 输出 1，否则 0），这就是 **1958 年的感知器**——本质是一个**线性分类器**：$\mathbf{w}^\top\mathbf{x}+b=0$ 是一条直线（超平面），把空间分成两半。它能学会的，只有**线性可分**的问题（如 AND、OR）。
With a step activation (1 if >0, else 0), this is the **1958 perceptron** — essentially a **linear classifier**: $\mathbf{w}^\top\mathbf{x}+b=0$ is a line (hyperplane) splitting space in two. It can only learn **linearly separable** problems (like AND, OR).

**感知器学习规则**：每次预测错了，就把权重往"修正这个错误"的方向调一点：$\mathbf{w} \leftarrow \mathbf{w} + \eta(y-\hat y)\mathbf{x}$。能分开就一定收敛（感知器收敛定理）。下面从零实现。
**The perceptron learning rule:** on each mistake, nudge the weights toward fixing it: $\mathbf{w} \leftarrow \mathbf{w} + \eta(y-\hat y)\mathbf{x}$. If separable, it's guaranteed to converge (the perceptron convergence theorem). We implement it below.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(0)

def perceptron(X, y, lr=0.1, epochs=20):
    w = np.zeros(X.shape[1]); b = 0.0
    for _ in range(epochs):
        for xi, yi in zip(X, y):
            pred = 1 if (xi @ w + b) > 0 else 0          # 阶跃激活 / step activation
            err = yi - pred                               # 预测错误量(0/±1)
            w += lr * err * xi                            # 感知器学习规则: 错了就往修正方向调
            b += lr * err
    return w, b

# AND 是线性可分的 → 感知器能学会 / AND is linearly separable
X_and = np.array([[0,0],[0,1],[1,0],[1,1]]); y_and = np.array([0,0,0,1])
w, b = perceptron(X_and, y_and)
preds = [(1 if (xi@w+b)>0 else 0) for xi in X_and]
print(f"AND 问题(线性可分): 感知器预测 {preds}, 真实 {list(y_and)} → {'全对!' if preds==list(y_and) else '错'}")
print("单个感知器能学会线性可分的 AND/OR — 它就是一个线性分类器")


<a id="2"></a>
## 2. 感知器学不会 XOR ⭐ / The Perceptron Fails on XOR

**XOR（异或）** 是感知器的滑铁卢，也是深度学习史上的著名转折点。XOR 的真值表：(0,0)→0, (0,1)→1, (1,0)→1, (1,1)→0。把这四个点画在平面上，你会发现：**没有任何一条直线能把"输出1"和"输出0"分开**——它们是线性不可分的。
**XOR** is the perceptron's Waterloo and a famous turning point in DL history. XOR's truth table: (0,0)→0, (0,1)→1, (1,0)→1, (1,1)→0. Plot these four points and you'll see: **no single line can separate the "output 1" from the "output 0" points** — they're not linearly separable.

既然单个感知器只能画一条直线，它就**永远学不会 XOR**（Minsky & Papert 1969 的这个证明一度让神经网络研究陷入"寒冬")。
Since a single perceptron only draws one line, it can **never learn XOR** (Minsky & Papert's 1969 proof triggered an "AI winter").


In [ ]:
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]]); y_xor = np.array([0,1,1,0])
w, b = perceptron(X_xor, y_xor, epochs=100)              # 训 100 轮也没用
preds = [(1 if (xi@w+b)>0 else 0) for xi in X_xor]
print(f"XOR 问题(线性不可分): 感知器预测 {preds}, 真实 {list(y_xor)} → {'全对' if preds==list(y_xor) else '学不会!'}")

fig, ax = plt.subplots(figsize=(5, 5))
for xi, yi in zip(X_xor, y_xor):
    ax.scatter(*xi, c="C2" if yi==1 else "C3", s=300, marker="o" if yi==1 else "X", zorder=3)
ax.text(0.5, 1.15, "绿○=输出1, 红✗=输出0", ha="center")
ax.set_xlim(-0.4,1.4); ax.set_ylim(-0.4,1.4); ax.set_xlabel("x1"); ax.set_ylabel("x2")
ax.set_title("XOR: 没有任何一条直线能分开两类 → 单个感知器学不会")
plt.tight_layout(); plt.show()
print("→ 对角的两个点同类, 另一对角同类; 一条直线无法分开 → 线性不可分(感知器的死穴)")


<a id="3"></a>
## 3. MLP 解决 XOR ⭐ / MLP Solves XOR

解药是**加一个隐藏层**：让网络先把输入**变换到一个新空间**，在那个空间里 XOR 变得线性可分，再用输出层分开。这就是**多层感知器(MLP)**——输入层 → 隐藏层(带非线性激活) → 输出层。
The cure is **adding a hidden layer**: let the network first **transform inputs into a new space** where XOR becomes linearly separable, then separate with the output layer. This is the **multilayer perceptron (MLP)** — input → hidden (with nonlinear activation) → output.

直觉：隐藏层的每个神经元学一条"半平面"边界，组合起来就能围出 XOR 需要的非线性决策区域。下面用 PyTorch 搭一个**2→4→1** 的小 MLP，它能轻松学会 XOR。
Intuition: each hidden neuron learns a "half-plane" boundary; combined, they carve out the nonlinear region XOR needs. Below we build a tiny **2→4→1** MLP in PyTorch — it learns XOR easily.


In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(0)

Xt = torch.tensor(X_xor, dtype=torch.float32)
yt = torch.tensor(y_xor, dtype=torch.float32).reshape(-1, 1)

# 2 输入 → 4 隐藏(带 ReLU 非线性) → 1 输出 / a tiny MLP with one hidden layer
mlp = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1))
opt = torch.optim.Adam(mlp.parameters(), lr=0.1)
loss_fn = nn.BCEWithLogitsLoss()                         # 二分类损失(含 sigmoid)
for _ in range(500):
    opt.zero_grad(); loss = loss_fn(mlp(Xt), yt); loss.backward(); opt.step()

with torch.no_grad():
    preds = (torch.sigmoid(mlp(Xt)) > 0.5).int().ravel().tolist()
print(f"MLP(2→4→1) 学 XOR: 预测 {preds}, 真实 {list(y_xor)} → {'全对!' if preds==list(y_xor) else '错'}")
print("加一个隐藏层 + 非线性激活 → 把输入变换到 XOR 线性可分的新空间 → 解决!")
print("这就是深度学习的核心: 多层非线性变换能拟合单层做不到的复杂函数")


<a id="4"></a>
## 4. 为什么必须非线性激活 ⭐ / Why Nonlinear Activations Are Mandatory

一个深刻且高频的考点：**如果隐藏层不加非线性激活，再多层也等于一层。** 因为线性变换的复合还是线性变换：$\mathbf{W}_2(\mathbf{W}_1\mathbf{x}) = (\mathbf{W}_2\mathbf{W}_1)\mathbf{x}$——两个线性层叠起来等价于一个线性层。**非线性激活(ReLU/sigmoid…)才是让深度网络真正"深"、能表达复杂函数的关键。**
A deep and frequently-tested point: **without nonlinear activations, any number of layers collapses to one.** Because composing linear maps stays linear: $\mathbf{W}_2(\mathbf{W}_1\mathbf{x}) = (\mathbf{W}_2\mathbf{W}_1)\mathbf{x}$ — two linear layers equal one. **Nonlinear activations (ReLU/sigmoid…) are what make a deep net truly deep and able to express complex functions.**

下面验证：去掉 ReLU 的"深层"网络在 XOR 上和单个感知器一样无能。
We verify: a "deep" net with ReLU removed is as helpless on XOR as a single perceptron.


In [ ]:
# 去掉激活函数的 "深层" 网络: 两个线性层直接相连 / a "deep" net with NO activation
linear_net = nn.Sequential(nn.Linear(2, 8), nn.Linear(8, 1))   # 注意: 没有 ReLU!
opt = torch.optim.Adam(linear_net.parameters(), lr=0.1)
for _ in range(500):
    opt.zero_grad(); loss = loss_fn(linear_net(Xt), yt); loss.backward(); opt.step()
with torch.no_grad():
    preds = (torch.sigmoid(linear_net(Xt)) > 0.5).int().ravel().tolist()
print(f"无激活的'深层'网 学 XOR: 预测 {preds}, 真实 {list(y_xor)} → {'全对' if preds==list(y_xor) else '学不会!'}")
print("→ 没有非线性激活, 8 个神经元的两层网络 = 一个线性层 = 还是学不会 XOR")
print("结论: 线性层复合仍是线性(W₂W₁x); 非线性激活才让深度网络真正'深'")


<a id="5"></a>
## 5. MLP on Digits + 小结 / MLP on Digits & Summary

最后在真实任务上跑 MLP：**Digits**（8×8 手写数字，10 类，MNIST 的迷你版）。一个简单的 MLP 就能达到很高准确率——这就是深度学习的起点。(后面 9.2 会从零实现反向传播，9.3 用 PyTorch 正式训练。)
Finally, an MLP on a real task: **Digits** (8×8 handwritten digits, 10 classes, a mini MNIST). A simple MLP already reaches high accuracy — the starting point of deep learning. (9.2 implements backprop from scratch; 9.3 trains properly in PyTorch.)


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
Xd = digits.data / 16.0                                  # 归一化到 [0,1] / scale
X_tr, X_te, y_tr, y_te = train_test_split(Xd, digits.target, test_size=0.3, stratify=digits.target, random_state=0)
Xtr_t = torch.tensor(X_tr, dtype=torch.float32); ytr_t = torch.tensor(y_tr)
Xte_t = torch.tensor(X_te, dtype=torch.float32)

# 64 → 64(ReLU) → 32(ReLU) → 10 的 MLP / a small MLP for 10-class digits
net = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
opt = torch.optim.Adam(net.parameters(), lr=1e-2)
ce = nn.CrossEntropyLoss()                               # 多分类交叉熵(自带 softmax)
for epoch in range(150):
    opt.zero_grad(); loss = ce(net(Xtr_t), ytr_t); loss.backward(); opt.step()
acc = (net(Xte_t).argmax(1).numpy() == y_te).mean()
print(f"MLP(64→64→32→10) on Digits: test 准确率 = {acc:.3f}")
print("一个简单 MLP 就能很好地分类手写数字 → 深度学习从这里开始")


```
神经元: a = σ(wᵀx+b); 感知器 = 阶跃激活的神经元 = 线性分类器
感知器死穴: 只能画一条直线 → 学不会 XOR(线性不可分) → 触发 AI 寒冬
MLP: 输入→隐藏层(非线性激活)→输出; 隐藏层把输入变换到线性可分的新空间 → 解决 XOR
为什么必须非线性激活: 线性层复合仍是线性(W₂W₁x), 没激活=单层; 激活让网络真正"深"
万能逼近定理: 含一个隐藏层的 MLP 可逼近任意连续函数(够宽的话)
```

### 💡 面试速查 / Interview cheat-sheet
1. **感知器=线性分类器**, 学不会 XOR(线性不可分)。
   Perceptron = linear classifier; can't learn XOR (not linearly separable).
2. **MLP 加隐藏层** → 把输入变换到可分空间 → 解决 XOR。
   MLP's hidden layer transforms inputs into a separable space → solves XOR.
3. **必须非线性激活**: 否则多层=单层(线性复合仍线性)。
   Nonlinear activations are mandatory; otherwise many layers = one (linear composition stays linear).
4. **万能逼近定理**: 一个隐藏层(够宽)即可逼近任意连续函数。
   Universal approximation: one (wide enough) hidden layer can approximate any continuous function.
5. 一个神经元 = 加权和 + 偏置 + 激活。
   A neuron = weighted sum + bias + activation.

### 下一节 / Next
**9.2 神经网络从零实现**——MLP 怎么训练? 用纯 NumPy 手写前向传播、反向传播、梯度下降, 彻底搞懂 backprop。
**9.2 NN from Scratch** — how is an MLP trained? Hand-code forward pass, backpropagation, and gradient descent in pure NumPy to truly understand backprop.
